In [ ]:

n_training_samples = len(X_test)
batch_size = 8
diff = 0
n_test_samples = 100
motifs_found_caller_arr = []
motif_errs_caller_arr = []
motifs_found_caller_arr_ = []
motif_errs_caller_arr_ = []
motifs_found_search_arr = []
motif_errs_search_arr = []
model = model.to(device)

with torch.no_grad():
    for ind in tqdm(range(0, n_training_samples, batch_size)):

        if n_training_samples - ind < batch_size:
            continue
        
        input_seqs = [
            normalize([X_test[k]], norm='max').flatten() for k in range(ind, ind + batch_size)]
        
        target_seqs = y_test[ind: ind + batch_size]

        input_seqs = pad_sequence([torch.tensor(
                    i, dtype=torch.float32) for i in input_seqs], batch_first=True)
        target_seqs = pad_sequence([torch.tensor(
                    i, dtype=torch.float32) for i in target_seqs], batch_first=True)
        
        input_seqs = input_seqs.view(input_seqs.shape[0], 1, input_seqs.shape[1])
        #input_seqs = input_seqs.to(device)
        
        pad_length_input = input_seqs.shape[2]
        n_samples = input_seqs.shape[0]

        pad_length_target = target_seqs.shape[1]

        model_output = model(input_seqs)
        model_output = model_output.permute(1, 0, 2)  # Assuming log probs are computed in network
        
        
        n_timesteps = model_output.shape[0]
        input_lengths = torch.tensor([n_timesteps for i in range(n_samples)])
        label_lengths = torch.tensor([len(y_test[ind + i]) for i in range(n_samples)])
        
        loss = ctc(
            log_probs=model_output, targets=target_seqs, input_lengths=input_lengths, target_lengths=label_lengths)
        
        
        model_output = model_output.permute(1, 0, 2).detach().cpu()
        #print(model_output.shape)

        for k in range(batch_size):
            original = payloads_test[ind + k]
            greedy_result = greedy_decoder(model_output[k])
            greedy_transcript = " ".join(greedy_result)
            beam_transcript = torch_ctc(
                n_classes=19, model_output=model_output[k].view(1, model_output[k].shape[0], 19), beam_width=50)
            actual_transcript = " ".join([str(i) for i in y_test[ind + k]])
            sorted_greedy = sort_transcript(greedy_transcript)
            sorted_beam = sort_transcript(beam_transcript)
            sorted_actual = sort_transcript(actual_transcript)
            
            print(sorted_greedy)
            print(sorted_beam)
            print(sorted_actual)
            print(original)

            motifs_found_caller, motif_errs_caller = evaluate_prediction(
                sorted_greedy, original)
            print(motifs_found_caller)
            print(motif_errs_caller)
            motifs_found_caller_, motif_errs_caller_ = evaluate_prediction(
                sorted_beam, original)
            print(motifs_found_caller_)
            print(motif_errs_caller_)
            print()
            motifs_found_caller_arr.append(motifs_found_caller)
            motif_errs_caller_arr.append(motif_errs_caller)
            
            motifs_found_search, motif_errs_search = evaluate_prediction(
                sorted_beam, original)
            motifs_found_search_arr.append(motifs_found_search)
            motif_errs_search_arr.append(motif_errs_search)

            
        torch.cuda.empty_cache()
        """
        if ind >= n_test_samples:
            print(diff / (n_test_samples))
            break
        """